# Titanic KNN - Simple Notebook
This notebook trains a K-Nearest Neighbors classifier on the Titanic dataset and saves the model artifacts to `../models/`.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pickle
import os

# Load data
csv_path = '../data/titanic.csv'
if not os.path.exists(csv_path):
    csv_path = '../titanic.csv'

df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]

# Drop extraneous zero columns if present
zero_cols = [c for c in df.columns if c.lower().startswith('zero') or c.lower() == 'zero']
if zero_cols:
    df = df.drop(columns=zero_cols)

# Standardize column names cleanly
rename_map = {}
for c in df.columns:
    lc = c.lower()
    if lc == 'passengerid':
        rename_map[c] = 'PassengerId'
    elif lc == 'age':
        rename_map[c] = 'Age'
    elif lc == 'fare':
        rename_map[c] = 'Fare'
    elif lc == 'sex':
        rename_map[c] = 'Sex'
    elif lc == 'sibsp':
        rename_map[c] = 'SibSp'
    elif lc == 'parch':
        rename_map[c] = 'Parch'
    elif lc == 'pclass':
        rename_map[c] = 'Pclass'
    elif lc == 'embarked':
        rename_map[c] = 'Embarked'
    elif any(k in lc for k in ['surv', 'urv', 'vive', 'surviv']):
        rename_map[c] = 'Survived'

if rename_map:
    df = df.rename(columns=rename_map)

# Convert target to numeric
try:
    df['Survived'] = pd.to_numeric(df['Survived'])
except Exception:
    df['Survived'] = df['Survived'].apply(lambda x: 1 if str(x).strip().lower() in ('1', 'yes', 'true', 'y', 'survived') else 0)

# Minimal preprocessing

def preprocess(frame, remove_outliers=False):
    data = frame.copy()
    data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])
    data['Age'] = data['Age'].fillna(data['Age'].median())

    # Preserve numeric codes already present in the CSV.
    # If the source ever contains strings, map them safely to the same codes.
    if data['Sex'].dtype == object:
        data['Sex'] = data['Sex'].map(lambda x: 1 if str(x).lower() == 'female' else 0)
    if data['Embarked'].dtype == object:
        data['Embarked'] = data['Embarked'].map(lambda x: {'S': 0, 'C': 1, 'Q': 2}.get(str(x).upper(), 0))

    if remove_outliers:
        for col in ['Age', 'Fare']:
            q1 = data[col].quantile(0.25)
            q3 = data[col].quantile(0.75)
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            data = data[(data[col] >= lower) & (data[col] <= upper)]

    return data


def train_and_evaluate(frame, remove_outliers=False):
    data = preprocess(frame, remove_outliers=remove_outliers)

    X = data[['Age', 'Fare', 'Sex', 'SibSp', 'Parch', 'Pclass', 'Embarked']].copy()
    y = data['Survived'].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    param_grid = {
        'n_neighbors': [3, 5, 7, 9],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    }

    grid = GridSearchCV(
        KNeighborsClassifier(),
        param_grid,
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )
    grid.fit(X_train_s, y_train)
    best_model = grid.best_estimator_

    y_pred = best_model.predict(X_test_s)
    metrics = {
        'rows': len(data),
        'best_params': grid.best_params_,
        'cv_accuracy': grid.best_score_,
        'test_accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'model': best_model,
        'scaler': scaler,
        'feature_names': list(X.columns)
    }
    return metrics

# Compare baseline vs improved setup
baseline = train_and_evaluate(df, remove_outliers=True)
improved = train_and_evaluate(df, remove_outliers=False)

comparison = pd.DataFrame([
    {
        'setup': 'With outlier removal',
        'rows': baseline['rows'],
        'cv_accuracy': baseline['cv_accuracy'],
        'test_accuracy': baseline['test_accuracy'],
        'precision': baseline['precision'],
        'recall': baseline['recall'],
        'f1': baseline['f1']
    },
    {
        'setup': 'No outlier removal',
        'rows': improved['rows'],
        'cv_accuracy': improved['cv_accuracy'],
        'test_accuracy': improved['test_accuracy'],
        'precision': improved['precision'],
        'recall': improved['recall'],
        'f1': improved['f1']
    }
])

print('=== METRIC COMPARISON ===')
print(comparison)

best = improved if improved['test_accuracy'] >= baseline['test_accuracy'] else baseline
print('\nSelected setup:', 'No outlier removal' if best is improved else 'With outlier removal')
print('Best params:', best['best_params'])
print('Best CV accuracy:', best['cv_accuracy'])
print('Best test accuracy:', best['test_accuracy'])
print('Best confusion matrix:\n', best['confusion_matrix'])

# Save chosen artifacts for app.py
os.makedirs('../models', exist_ok=True)
with open('../models/knn_model.pkl', 'wb') as f:
    pickle.dump(best['model'], f)
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(best['scaler'], f)
with open('../models/feature_names.pkl', 'wb') as f:
    pickle.dump(best['feature_names'], f)

print('\nSaved artifacts to ../models')
print('Feature columns:', best['feature_names'])

=== METRIC COMPARISON ===
                  setup  rows  cv_accuracy  test_accuracy  precision  \
0  With outlier removal  1055     0.786778       0.772512    0.50000   
1    No outlier removal  1309     0.774555       0.782443    0.59322   

     recall        f1  
0  0.333333  0.400000  
1  0.514706  0.551181  

Selected setup: No outlier removal
Best params: {'metric': 'euclidean', 'n_neighbors': 9, 'weights': 'uniform'}
Best CV accuracy: 0.7745545682387788
Best test accuracy: 0.7824427480916031
Best confusion matrix:
 [[170  24]
 [ 33  35]]

Saved artifacts to ../models
Feature columns: ['Age', 'Fare', 'Sex', 'SibSp', 'Parch', 'Pclass', 'Embarked']
